## Preprocess SFO NOAA Data

This notebook loads the combined parquet, inspects missing values, cleans dtypes, and saves to a final cleaned parquet.

In [1]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parents[1]

COMBINED_PARQUET_PATH = REPO_ROOT / "Data_Extraction" / "santi" / "data" / "combined" / "sfo_noaa_combined.parquet"
CLEANED_DATA_DIR = REPO_ROOT / "Data_Preprocessing" / "santi" / "data" / "cleaned"
CLEANED_PARQUET_PATH = CLEANED_DATA_DIR / "sfo_noaa_cleaned.parquet"

CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

### Load combined data

In [2]:
weather_df = pd.read_parquet(COMBINED_PARQUET_PATH)

display(weather_df)

,STATION,Station_name,DATE,Year,Month,Day,Hour,Minute,LATITUDE,LONGITUDE,...,precipitation_24_hour_Quality_Code,precipitation_24_hour_Report_Type,precipitation_24_hour_Source_Code,precipitation_24_hour_Source_Station_ID,REM,REM_Measurement_Code,REM_Quality_Code,REM_Report_Type,REM_Source_Code,REM_Source_Station_ID
0,USW00023234,SAN FRANCISCO INTL AP,2022-01-01 00:00:00,2022,1,1,0,0,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,SYN08072494 32566 42915 10117 20044 30094 4012...,NaN,NaN,FM12,223.0,ICAO-KSFO
1,USW00023234,SAN FRANCISCO INTL AP,2022-01-01 00:56:00,2022,1,1,0,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,MET09812/31/21 16:56:03 METAR KSFO 010056Z 300...,NaN,NaN,FM15,343.0,724940-23234
2,USW00023234,SAN FRANCISCO INTL AP,2022-01-01 01:56:00,2022,1,1,1,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,MET09812/31/21 17:56:03 METAR KSFO 010156Z 300...,NaN,NaN,FM15,343.0,724940-23234
3,USW00023234,SAN FRANCISCO INTL AP,2022-01-01 02:56:00,2022,1,1,2,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,MET10112/31/21 18:56:03 METAR KSFO 010256Z 290...,NaN,NaN,FM15,343.0,724940-23234
4,USW00023234,SAN FRANCISCO INTL AP,2022-01-01 03:56:00,2022,1,1,3,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,MET09812/31/21 19:56:03 METAR KSFO 010356Z 310...,NaN,NaN,FM15,343.0,724940-23234
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89171,USW00023234,SAN FRANCISCO INTL AP,2026-06-09 14:21:00,2026,6,9,14,21,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,KSFO 091421Z 34006KT 10SM FEW006 BKN013 OVC043...,NaN,NaN,FM16,413.0,ICAO-KSFO
89172,USW00023234,SAN FRANCISCO INTL AP,2026-06-09 14:56:00,2026,6,9,14,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,KSFO 091456Z 33007KT 10SM FEW006 BKN013 OVC045...,NaN,NaN,FM15,413.0,ICAO-KSFO
89173,USW00023234,SAN FRANCISCO INTL AP,2026-06-09 15:43:00,2026,6,9,15,43,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,KSFO 091543Z 33007KT 10SM FEW006 SCT014 BKN039...,NaN,NaN,FM16,413.0,ICAO-KSFO
89174,USW00023234,SAN FRANCISCO INTL AP,2026-06-09 15:56:00,2026,6,9,15,56,37.6197,-122.3656,...,NaN,NaN,NaN,NaN,KSFO 091556Z 33006KT 10SM FEW006 SCT014 BKN039...,NaN,NaN,FM15,413.0,ICAO-KSFO


### Missing values by column

In [3]:
nan_count_by_column = (
    weather_df.isna()
    .sum()
    .rename("nan_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
nan_count_by_column["nan_percent"] = (
    nan_count_by_column["nan_count"] / len(weather_df) * 100
).round(2)

display(nan_count_by_column.sort_values("nan_count", ascending=False))

,column,nan_count,nan_percent
325,REM_Quality_Code,89176,100.0
324,REM_Measurement_Code,89176,100.0
306,precipitation_18_hour_Measurement_Code,89176,100.0
305,precipitation_18_hour,89176,100.0
304,precipitation_15_hour_Source_Station_ID,89176,100.0
...,...,...,...
6,Hour,0,0.0
5,Day,0,0.0
4,Month,0,0.0
1,Station_name,0,0.0


### Select columns

In [4]:
SELECTED_COLUMNS = [
    "DATE",
    "LATITUDE",
    "LONGITUDE",
    "ELEVATION",
    "temperature",
    "dew_point_temperature",
    "relative_humidity",
    "visibility",
    "wind_speed",
    "wind_direction",
    "sea_level_pressure",
    "station_level_pressure",
    "wet_bulb_temperature",
    "ceiling_height",
    "altimeter",
    "sky_condition_baseht",
    "sky_cover_summation_baseht_1",
    "precipitation",
]

missing_columns = [column for column in SELECTED_COLUMNS if column not in weather_df.columns]

selected_weather_df = weather_df.loc[:, SELECTED_COLUMNS].copy()

display(selected_weather_df)

,DATE,LATITUDE,LONGITUDE,ELEVATION,temperature,dew_point_temperature,relative_humidity,visibility,wind_speed,wind_direction,sea_level_pressure,station_level_pressure,wet_bulb_temperature,ceiling_height,altimeter,sky_condition_baseht,sky_cover_summation_baseht_1,precipitation
0,2022-01-01 00:00:00,37.6197,-122.3656,3.0,11.7,4.4,61.0,16.000,7.7,290.0,1012.4,1009.4,8.2,22000.0,NaN,800.0,NaN,NaN
1,2022-01-01 00:56:00,37.6197,-122.3656,3.0,10.6,5.0,69.0,16.093,7.2,300.0,1012.8,1012.3,7.9,22000.0,1012.9,610.0,610.0,0.0
2,2022-01-01 01:56:00,37.6197,-122.3656,3.0,10.0,4.4,68.0,16.093,5.1,300.0,1013.3,1012.6,7.4,22000.0,1013.2,579.0,579.0,0.0
3,2022-01-01 02:56:00,37.6197,-122.3656,3.0,9.4,4.4,71.0,16.093,5.1,290.0,1014.0,1013.6,7.1,22000.0,1014.2,NaN,NaN,0.0
4,2022-01-01 03:56:00,37.6197,-122.3656,3.0,9.4,5.0,74.0,16.093,5.1,310.0,1014.6,1014.0,7.3,22000.0,1014.6,914.0,914.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89171,2026-06-09 14:21:00,37.6197,-122.3656,3.0,15.6,13.9,90.0,16.090,3.1,340.0,NaN,1015.4,14.6,396.0,1015.9,NaN,183.0,NaN
89172,2026-06-09 14:56:00,37.6197,-122.3656,3.0,16.1,13.9,87.0,16.090,3.6,330.0,1016.0,1015.4,14.8,396.0,1015.9,NaN,183.0,NaN
89173,2026-06-09 15:43:00,37.6197,-122.3656,3.0,16.1,13.3,83.0,16.090,3.6,330.0,NaN,1015.4,14.5,1189.0,1015.9,NaN,183.0,NaN
89174,2026-06-09 15:56:00,37.6197,-122.3656,3.0,16.7,13.9,84.0,16.090,3.1,330.0,1015.8,1015.4,15.1,1189.0,1015.9,NaN,183.0,NaN


### Clean dtypes

In [5]:
cleaned_weather_df = selected_weather_df.copy()

cleaned_weather_df["DATE"] = pd.to_datetime(cleaned_weather_df["DATE"], errors="coerce")

numeric_columns = [column for column in SELECTED_COLUMNS if column != "DATE"]
for column in numeric_columns:
    cleaned_weather_df[column] = pd.to_numeric(cleaned_weather_df[column], errors="coerce")

cleaned_weather_df = cleaned_weather_df.sort_values("DATE", na_position="last").reset_index(drop=True)

dtype_summary = cleaned_weather_df.dtypes.rename("dtype").reset_index().rename(columns={"index": "column"})
display(dtype_summary)
display(cleaned_weather_df.head())

,column,dtype
0,DATE,datetime64[us]
1,LATITUDE,float64
2,LONGITUDE,float64
3,ELEVATION,float64
4,temperature,float64
5,dew_point_temperature,float64
6,relative_humidity,float64
7,visibility,float64
8,wind_speed,float64
9,wind_direction,float64


,DATE,LATITUDE,LONGITUDE,ELEVATION,temperature,dew_point_temperature,relative_humidity,visibility,wind_speed,wind_direction,sea_level_pressure,station_level_pressure,wet_bulb_temperature,ceiling_height,altimeter,sky_condition_baseht,sky_cover_summation_baseht_1,precipitation
0,2022-01-01 00:00:00,37.6197,-122.3656,3.0,11.7,4.4,61.0,16.000,7.7,290.0,1012.4,1009.4,8.2,22000.0,NaN,800.0,NaN,NaN
1,2022-01-01 00:56:00,37.6197,-122.3656,3.0,10.6,5.0,69.0,16.093,7.2,300.0,1012.8,1012.3,7.9,22000.0,1012.9,610.0,610.0,0.0
2,2022-01-01 01:56:00,37.6197,-122.3656,3.0,10.0,4.4,68.0,16.093,5.1,300.0,1013.3,1012.6,7.4,22000.0,1013.2,579.0,579.0,0.0
3,2022-01-01 02:56:00,37.6197,-122.3656,3.0,9.4,4.4,71.0,16.093,5.1,290.0,1014.0,1013.6,7.1,22000.0,1014.2,NaN,NaN,0.0
4,2022-01-01 03:56:00,37.6197,-122.3656,3.0,9.4,5.0,74.0,16.093,5.1,310.0,1014.6,1014.0,7.3,22000.0,1014.6,914.0,914.0,0.0


In [6]:
nan_count_by_column = (
    cleaned_weather_df.isna()
    .sum()
    .rename("nan_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
nan_count_by_column["nan_percent"] = (
    nan_count_by_column["nan_count"] / len(cleaned_weather_df) * 100
).round(2)

display(nan_count_by_column.sort_values("nan_count", ascending=False))

,column,nan_count,nan_percent
17,precipitation,56049,62.85
16,sky_cover_summation_baseht_1,50082,56.16
15,sky_condition_baseht,46714,52.38
14,altimeter,44887,50.34
13,ceiling_height,43701,49.01
12,wet_bulb_temperature,41565,46.61
11,station_level_pressure,41563,46.61
10,sea_level_pressure,40957,45.93
9,wind_direction,35410,39.71
8,wind_speed,35397,39.69


### Remove missing values

In [7]:
cleaned_weather_df = cleaned_weather_df.dropna(subset=["temperature"])
display(cleaned_weather_df)

,DATE,LATITUDE,LONGITUDE,ELEVATION,temperature,dew_point_temperature,relative_humidity,visibility,wind_speed,wind_direction,sea_level_pressure,station_level_pressure,wet_bulb_temperature,ceiling_height,altimeter,sky_condition_baseht,sky_cover_summation_baseht_1,precipitation
0,2022-01-01 00:00:00,37.6197,-122.3656,3.0,11.7,4.4,61.0,16.000,7.7,290.0,1012.4,1009.4,8.2,22000.0,NaN,800.0,NaN,NaN
1,2022-01-01 00:56:00,37.6197,-122.3656,3.0,10.6,5.0,69.0,16.093,7.2,300.0,1012.8,1012.3,7.9,22000.0,1012.9,610.0,610.0,0.0
2,2022-01-01 01:56:00,37.6197,-122.3656,3.0,10.0,4.4,68.0,16.093,5.1,300.0,1013.3,1012.6,7.4,22000.0,1013.2,579.0,579.0,0.0
3,2022-01-01 02:56:00,37.6197,-122.3656,3.0,9.4,4.4,71.0,16.093,5.1,290.0,1014.0,1013.6,7.1,22000.0,1014.2,NaN,NaN,0.0
4,2022-01-01 03:56:00,37.6197,-122.3656,3.0,9.4,5.0,74.0,16.093,5.1,310.0,1014.6,1014.0,7.3,22000.0,1014.6,914.0,914.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89171,2026-06-09 14:21:00,37.6197,-122.3656,3.0,15.6,13.9,90.0,16.090,3.1,340.0,NaN,1015.4,14.6,396.0,1015.9,NaN,183.0,NaN
89172,2026-06-09 14:56:00,37.6197,-122.3656,3.0,16.1,13.9,87.0,16.090,3.6,330.0,1016.0,1015.4,14.8,396.0,1015.9,NaN,183.0,NaN
89173,2026-06-09 15:43:00,37.6197,-122.3656,3.0,16.1,13.3,83.0,16.090,3.6,330.0,NaN,1015.4,14.5,1189.0,1015.9,NaN,183.0,NaN
89174,2026-06-09 15:56:00,37.6197,-122.3656,3.0,16.7,13.9,84.0,16.090,3.1,330.0,1015.8,1015.4,15.1,1189.0,1015.9,NaN,183.0,NaN


In [8]:
cleaned_weather_df = cleaned_weather_df.dropna()
display(cleaned_weather_df)

,DATE,LATITUDE,LONGITUDE,ELEVATION,temperature,dew_point_temperature,relative_humidity,visibility,wind_speed,wind_direction,sea_level_pressure,station_level_pressure,wet_bulb_temperature,ceiling_height,altimeter,sky_condition_baseht,sky_cover_summation_baseht_1,precipitation
1,2022-01-01 00:56:00,37.6197,-122.3656,3.0,10.6,5.0,69.0,16.093,7.2,300.0,1012.8,1012.3,7.9,22000.0,1012.9,610.0,610.0,0.0
2,2022-01-01 01:56:00,37.6197,-122.3656,3.0,10.0,4.4,68.0,16.093,5.1,300.0,1013.3,1012.6,7.4,22000.0,1013.2,579.0,579.0,0.0
4,2022-01-01 03:56:00,37.6197,-122.3656,3.0,9.4,5.0,74.0,16.093,5.1,310.0,1014.6,1014.0,7.3,22000.0,1014.6,914.0,914.0,0.0
15,2022-01-01 12:56:00,37.6197,-122.3656,3.0,7.8,3.9,77.0,16.093,4.6,360.0,1019.6,1019.0,6.0,22000.0,1019.6,6096.0,6096.0,0.0
16,2022-01-01 13:56:00,37.6197,-122.3656,3.0,5.0,2.2,82.0,16.093,0.0,999.0,1020.8,1020.4,3.8,22000.0,1021.0,6096.0,6096.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44382,2025-08-27 02:56:00,37.6197,-122.3656,3.0,18.9,12.8,68.0,16.093,6.7,290.0,1014.0,1013.3,15.4,22000.0,1013.9,274.0,274.0,0.0
44384,2025-08-27 03:56:00,37.6197,-122.3656,3.0,17.8,12.8,73.0,16.093,6.2,280.0,1014.8,1014.3,14.9,22000.0,1014.9,244.0,244.0,0.0
44385,2025-08-27 04:56:00,37.6197,-122.3656,3.0,17.8,13.3,75.0,16.093,4.6,260.0,1015.1,1014.6,15.2,22000.0,1015.2,244.0,244.0,0.0
44386,2025-08-27 05:56:00,37.6197,-122.3656,3.0,17.2,12.8,76.0,16.093,4.1,280.0,1015.6,1015.0,14.7,22000.0,1015.6,244.0,244.0,0.0


### Summary check

In [9]:
nan_count_by_column = (
    cleaned_weather_df.isna()
    .sum()
    .rename("nan_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
nan_count_by_column["nan_percent"] = (
    nan_count_by_column["nan_count"] / len(cleaned_weather_df) * 100
).round(2)

display(nan_count_by_column)

,column,nan_count,nan_percent
0,DATE,0,0.0
1,LATITUDE,0,0.0
2,LONGITUDE,0,0.0
3,ELEVATION,0,0.0
4,temperature,0,0.0
5,dew_point_temperature,0,0.0
6,relative_humidity,0,0.0
7,visibility,0,0.0
8,wind_speed,0,0.0
9,wind_direction,0,0.0


### Save Cleaned Parquet

In [10]:
cleaned_weather_df.to_parquet(CLEANED_PARQUET_PATH, index=False)

print(f"Wrote cleaned NOAA data to: {CLEANED_PARQUET_PATH}")

Wrote cleaned NOAA data to: c:\Users\smeri\OneDrive\Desktop\MIDS Coursework\207-Summer26-FinalProject-MLModel\Data_Preprocessing\santi\data\cleaned\sfo_noaa_cleaned.parquet
